In [ ]:
import os
import json
import pdal
from concurrent.futures import ThreadPoolExecutor, as_completed

# Find all .copc.laz files
raw_dir = os.path.join("..", "data", "raw")
with os.scandir(raw_dir) as it:
    input_files = sorted(
        entry.path for entry in it
        if entry.is_file() and entry.name.endswith(".copc.laz")
    )
if not input_files:
    raise FileNotFoundError(f"No .copc.laz files found in {raw_dir}")
print(f"Found {len(input_files)} files")

fp_out = os.path.join("..", "data", "merged.laz")
tmp_dir = os.path.join("..", "data", "_merge_tmp")
os.makedirs(tmp_dir, exist_ok=True)


def stream_one(fp, idx):
    """Stream-copy one tile to a temp .laz. No Python-side point storage."""
    tmp_out = os.path.join(tmp_dir, f"part_{idx:05d}.laz")
    pipeline_json = {
        "pipeline": [
            {"type": "readers.copc", "filename": fp},
            {"type": "writers.las",
             "filename": tmp_out,
             "compression": "laszip",
             "forward": "all"},
        ]
    }
    p = pdal.Pipeline(json.dumps(pipeline_json))
    n = p.execute_streaming(chunk_size=1_000_000)
    return tmp_out, n


def merge_group(files, out_path):
    """Merge a list of .laz files into one .laz, streaming."""
    pipeline_json = {
        "pipeline": [
            *[{"type": "readers.las", "filename": f} for f in files],
            {"type": "filters.merge"},
            {"type": "writers.las",
             "filename": out_path,
             "compression": "laszip",
             "forward": "all"},
        ]
    }
    p = pdal.Pipeline(json.dumps(pipeline_json))
    return p.execute_streaming(chunk_size=1_000_000)


# --- Stage 1: parallel read+write of source tiles to per-tile temp files ---
max_workers = min(len(input_files), 16)
tmp_files = []
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(stream_one, fp, i): fp for i, fp in enumerate(input_files)}
    for fut in as_completed(futures):
        tmp_out, n = fut.result()
        tmp_files.append(tmp_out)
        print(f"Streamed {n:,} points from {os.path.basename(futures[fut])}")

# --- Stage 2: parallel merge of groups ---
n_groups = 4
# Round-robin split so groups are roughly equal in size even if tiles vary
groups = [tmp_files[i::n_groups] for i in range(n_groups)]
group_outputs = [os.path.join(tmp_dir, f"group_{i:02d}.laz") for i in range(n_groups)]

print(f"Merging {len(tmp_files)} files in {n_groups} parallel groups...")
with ThreadPoolExecutor(max_workers=n_groups) as ex:
    futures = {
        ex.submit(merge_group, g, out): out
        for g, out in zip(groups, group_outputs)
    }
    for fut in as_completed(futures):
        n = fut.result()
        print(f"  Merged group -> {os.path.basename(futures[fut])} ({n:,} points)")

# --- Stage 3: final merge of group outputs ---
print(f"Final merge of {n_groups} group files into {fp_out}...")
n_points = merge_group(group_outputs, fp_out)
print(f"Wrote {n_points:,} points to {fp_out}")

# --- Cleanup ---
for f in tmp_files + group_outputs:
    try:
        os.remove(f)
    except FileNotFoundError:
        pass
os.rmdir(tmp_dir)
print("Cleaned up temp files")

Found 37 files
Streamed 19,303,634 points from LHD_FXX_0849_6330_PTS_LAMB93_IGN69.copc.laz
Streamed 22,927,576 points from LHD_FXX_0847_6330_PTS_LAMB93_IGN69.copc.laz
Streamed 23,062,619 points from LHD_FXX_0846_6331_PTS_LAMB93_IGN69.copc.laz
Streamed 23,289,822 points from LHD_FXX_0848_6331_PTS_LAMB93_IGN69.copc.laz
Streamed 22,646,443 points from LHD_FXX_0847_6331_PTS_LAMB93_IGN69.copc.laz
Streamed 24,039,236 points from LHD_FXX_0848_6330_PTS_LAMB93_IGN69.copc.laz
Streamed 23,675,855 points from LHD_FXX_0849_6352_PTS_LAMB93_IGN69.copc.laz
Streamed 23,584,390 points from LHD_FXX_0859_6337_PTS_LAMB93_IGN69.copc.laz
Streamed 23,154,510 points from LHD_FXX_0856_6347_PTS_LAMB93_IGN69.copc.laz
Streamed 24,331,298 points from LHD_FXX_0849_6328_PTS_LAMB93_IGN69.copc.laz
Streamed 24,502,039 points from LHD_FXX_0859_6336_PTS_LAMB93_IGN69.copc.laz
Streamed 24,601,836 points from LHD_FXX_0856_6341_PTS_LAMB93_IGN69.copc.laz
Streamed 25,338,695 points from LHD_FXX_0856_6340_PTS_LAMB93_IGN69.copc.l

In [ ]:
import os
import json
import pdal
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

# ── Config ──
RAW_DIR = os.path.join("..", "data", "raw")
FP_OUT = os.path.join("..", "data", "merged.copc.laz")  # Keep COPC format
TMP_DIR = os.path.join("..", "data", "_merge_tmp")
CHUNK_SIZE = 5_000_000  # Larger chunks = less overhead
os.makedirs(TMP_DIR, exist_ok=True)

# ── Stage 0: Fast file discovery ──
input_files = sorted(
    os.path.join(RAW_DIR, f) 
    for f in os.listdir(RAW_DIR) 
    if f.endswith(".copc.laz")
)
if not input_files:
    raise FileNotFoundError(f"No .copc.laz files in {RAW_DIR}")
print(f"Found {len(input_files)} files")

# ── Stage 1: Parallel re-encode to aligned temp COPC tiles ──
def reencode(args):
    """Re-encode a COPC tile to standard LAZ with spatial bounds preserved."""
    fp, idx = args
    tmp_out = os.path.join(TMP_DIR, f"part_{idx:05d}.laz")
    
    # Use COPC reader but write standard LAZ (better merge performance)
    pipeline = {
        "pipeline": [
            {"type": "readers.copc", "filename": fp, "bounds": "([0, 10000000],[0, 10000000],[0, 10000])"},
            {"type": "writers.las",
             "filename": tmp_out,
             "compression": "laszip",
             "forward": "all",
             "minor_version": 4,
             "dataformat_id": 7}  # Ensure consistent format across tiles
        ]
    }
    
    p = pdal.Pipeline(json.dumps(pipeline))
    n = p.execute_streaming(chunk_size=CHUNK_SIZE)
    return tmp_out, n, fp

# Use processes instead of threads for true parallelism
max_workers = min(len(input_files), os.cpu_count() or 4)
tmp_files = []
print(f"Re-encoding {len(input_files)} files using {max_workers} processes...")

with ProcessPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(reencode, (fp, i)): fp for i, fp in enumerate(input_files)}
    for fut in as_completed(futures):
        tmp_out, n, orig = fut.result()
        tmp_files.append(tmp_out)
        print(f"  {os.path.basename(orig)}: {n:,} points → {tmp_out}")

# ── Stage 2: Single-pass streaming merge (no intermediate groups) ──
print(f"Merging {len(tmp_files)} files directly to output...")

# PDAL can handle many readers in one pipeline; avoid intermediate merges
readers = [{"type": "readers.las", "filename": f} for f in tmp_files]

merge_pipeline = {
    "pipeline": [
        *readers,
        {"type": "filters.merge"},
        {"type": "writers.copc",  # Output as COPC, not plain LAZ
         "filename": FP_OUT,
         "forward": "all",
         "compression": "laszip"}
    ]
}

p = pdal.Pipeline(json.dumps(merge_pipeline))
n_points = p.execute_streaming(chunk_size=CHUNK_SIZE * 2)  # Larger chunks for merge
print(f"Wrote {n_points:,} points to {FP_OUT}")

# ── Cleanup ──
import shutil
shutil.rmtree(TMP_DIR, ignore_errors=True)
print("Cleaned up")